In [1]:
from pathlib import Path
import os, glob, re
import numpy as np
import pandas as pd
from biopandas.pdb import PandasPdb
import MDAnalysis as mda
from rdkit import Chem
import tempfile
import subprocess

# =========================
# Paths
# =========================
RCSB_PROTEIN_DIR = Path("CLR-PDB-matched-to-opm_ids")
UNLABELED_DIR = Path("CLR-Unlabeled-Distinct-matched-to-opm_ids")
OPM_DIR = Path("OPM_PDB")

def get_protein_name(filename):
    basename = os.path.basename(filename)
    match = re.match(r'([a-zA-Z0-9]{4})', basename)
    return match.group(1).upper() if match else None


def get_mode_index(filename):
    basename = os.path.basename(filename)
    match = re.search(r'mode_(\d+)', basename)
    return int(match.group(1)) if match else None


def natural_sort_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', str(s))]


def grid_list(atom_df):
    return list(zip(atom_df['x_coord'], atom_df['y_coord'], atom_df['z_coord']))


def filtering_proteins(atom_df, grid_list, radius=5.0):
    import numpy as np

    # Adjust these column names if needed for your dataframe
    residue_cols = ['chain_id', 'residue_name', 'residue_number', 'insertion']
    residue_cols = [col for col in residue_cols if col in atom_df.columns]

    if not residue_cols:
        raise ValueError("Could not identify residue columns in atom_df.")

    atom_coords = atom_df[['x_coord', 'y_coord', 'z_coord']].values
    initially_filtered_atoms = set()

    # Step 1: find atoms within radius of any grid point
    for x, y, z in grid_list:
        distances_sq = (
            (atom_coords[:, 0] - x) ** 2 +
            (atom_coords[:, 1] - y) ** 2 +
            (atom_coords[:, 2] - z) ** 2
        )
        mask = distances_sq <= radius ** 2
        initially_filtered_atoms.update(atom_df.index[mask])

    print(f"Total atoms within {radius} Å cutoff: {len(initially_filtered_atoms)}")

    if not initially_filtered_atoms:
        return atom_df.loc[list(initially_filtered_atoms)]

    grouped = atom_df.groupby(residue_cols)

    # Step 2: residues passing the 50% rule
    residue_keep_set = set()

    for residue_key, residue_df in grouped:
        residue_atom_indices = set(residue_df.index)
        n_total = len(residue_atom_indices)
        n_filtered = len(residue_atom_indices & initially_filtered_atoms)

        if n_total == 0:
            continue

        fraction_present = n_filtered / n_total

        if fraction_present >= 0.5:
            residue_keep_set.add(residue_key)

    # Step 3: expand to all atoms in kept residues
    filtered_atoms = set()
    for residue_key, residue_df in grouped:
        if residue_key in residue_keep_set:
            filtered_atoms.update(residue_df.index)

    # Fallback: if nothing survives 50% rule, keep residues that had any atom in cutoff
    if len(filtered_atoms) == 0:
        print("No residues passed the 50% occupancy filter. Falling back to residues with at least one atom within cutoff.")

        fallback_residue_keep_set = set()

        for residue_key, residue_df in grouped:
            residue_atom_indices = set(residue_df.index)
            if len(residue_atom_indices & initially_filtered_atoms) > 0:
                fallback_residue_keep_set.add(residue_key)

        for residue_key, residue_df in grouped:
            if residue_key in fallback_residue_keep_set:
                filtered_atoms.update(residue_df.index)

    print(f"Total atoms after residue expansion + filtering: {len(filtered_atoms)}")
    return atom_df.loc[list(filtered_atoms)]


def read_protein_atoms(pdb_path):
    ppdb = PandasPdb().read_pdb(str(pdb_path))
    atoms = ppdb.df["ATOM"].copy()
    atoms = atoms[~atoms["atom_name"].str.startswith("H")].copy()
    return atoms


def read_clr_atoms(pdb_path):
    ppdb = PandasPdb().read_pdb(str(pdb_path))
    hetatm = ppdb.df["HETATM"].copy()
    clr = hetatm[hetatm["residue_name"] == "CLR"].copy()
    return clr


def atom_match_key(df):
    """
    Match RCSB atoms to OPM atoms by residue number, chain, and atom subtype/name.
    """
    return list(zip(
        df["residue_number"].astype(int),
        df["chain_id"].astype(str),
        df["atom_name"].astype(str).str.strip()
    ))


def map_rcsb_filtered_atoms_to_opm(filtered_rcsb_atoms, opm_atoms):
    """
    Takes atoms filtered using RCSB/Vina coordinate system and returns
    matching atoms from OPM coordinate system.
    """
    filtered_keys = set(atom_match_key(filtered_rcsb_atoms))

    opm_atoms = opm_atoms.copy()
    opm_atoms["_match_key"] = atom_match_key(opm_atoms)

    mapped = opm_atoms[opm_atoms["_match_key"].isin(filtered_keys)].copy()
    mapped = mapped.drop(columns=["_match_key"])

    return mapped


def save_atoms_as_pdb(atom_df, output_path):
    filtered_pdb = PandasPdb()
    filtered_pdb.df["ATOM"] = atom_df.copy()
    filtered_pdb.to_pdb(
        path=str(output_path),
        records=None,
        gz=False,
        append_newline=True
    )


def get_all_opm_clr_gridlists(opm_file):
    ligand = read_clr_atoms(opm_file)
    all_ligands = []

    for residue_number, chain_id in set(zip(ligand["residue_number"], ligand["chain_id"])):
        clr_atoms = ligand[
            (ligand["residue_number"] == residue_number) &
            (ligand["chain_id"] == chain_id)
        ]

        if not clr_atoms.empty:
            all_ligands.append(grid_list(clr_atoms))

    return all_ligands


def check_if_unlabeled_is_positive(positive_grid_list, unlabeled_grid_list, cutoff=5.0):
    positive_grid_list = np.asarray(positive_grid_list, dtype=float)
    unlabeled_grid_list = np.asarray(unlabeled_grid_list, dtype=float)

    if positive_grid_list.size == 0 or unlabeled_grid_list.size == 0:
        return False

    positive_centroid = positive_grid_list.mean(axis=0)
    unlabeled_centroid = unlabeled_grid_list.mean(axis=0)

    distance = np.linalg.norm(positive_centroid - unlabeled_centroid)
    return distance <= cutoff

def make_atom_key(df):
    return list(zip(
        df['residue_number'].astype(int),
        df['chain_id'].astype(str),
        df['atom_name'].astype(str).str.strip()
    ))


def map_rcsb_atoms_to_opm(filtered_rcsb_atoms, opm_protein):
    filtered_keys = set(make_atom_key(filtered_rcsb_atoms))

    opm_protein = opm_protein.copy()
    opm_protein['_key'] = make_atom_key(opm_protein)

    mapped_atoms = opm_protein[opm_protein['_key'].isin(filtered_keys)].copy()
    mapped_atoms = mapped_atoms.drop(columns=['_key'])

    return mapped_atoms

/home/alexhernandez/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def lookup_charge(chain, resseq, atomname, charges_full, charges_nochain, has_chain):
    # try chain-aware first
    if has_chain:
        val = charges_full.get((chain, resseq, atomname), None)
        if val is not None:
            return val

    # always try no-chain fallback too
    val = charges_nochain.get((resseq, atomname), None)
    if val is not None:
        return val

    return 0.0

def strip_dum_atoms(pdb_path, out_path):
    with open(pdb_path) as f_in, open(out_path, "w") as f_out:
        for line in f_in:
            if "DUM" not in line:
                f_out.write(line)


def run_pdb2pqr(clean_pdb, pqr_out, ph=7.4):
    cmd = [
        "pdb2pqr",
        "--ff=AMBER",
        f"--with-ph={ph}",
        "--quiet",
        str(clean_pdb),
        str(pqr_out),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0

def parse_pqr_charges(pqr_path):
    """
    Returns:
        charges_full[(chain, resseq, atomname)] = charge
        charges_nochain[(resseq, atomname)] = charge
        has_chain = bool
    """
    charges_full = {}
    charges_nochain = {}
    chain_values = set()

    with open(pqr_path) as f:
        for line in f:
            if not line.startswith(("ATOM", "HETATM")):
                continue
            try:
                atomname = line[12:16].strip()
                chain = line[21].strip()
                resseq = int(line[22:26].strip())
                charge = float(line[54:62].strip())
            except (ValueError, IndexError):
                continue

            chain_values.add(chain)
            charges_full[(chain, resseq, atomname)] = charge
            charges_nochain[(resseq, atomname)] = charge

    has_chain = any(c != "" for c in chain_values)
    return charges_full, charges_nochain, has_chain


def build_pqr_cache(full_pdb_dir, ph=7.4):
    full_pdb_dir = Path(full_pdb_dir)
    pdb_files = sorted(full_pdb_dir.glob("*.pdb"))

    pqr_cache = {}
    pqr_failures = []

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)

        for pdb_path in pdb_files:
            stem = pdb_path.stem
            match = re.match(r"([A-Za-z0-9]{4})", stem)

            if not match:
                pqr_failures.append(stem.upper())
                continue

            pdbid = match.group(1).upper()

            clean_pdb = tmpdir / f"{pdbid}_clean.pdb"
            pqr_out = tmpdir / f"{pdbid}.pqr"

            strip_dum_atoms(pdb_path, clean_pdb)

            ok = run_pdb2pqr(clean_pdb, pqr_out, ph=ph)

            if not ok or not pqr_out.exists():
                pqr_failures.append(pdbid)
                continue

            charges_full, charges_nochain, has_chain = parse_pqr_charges(pqr_out)
            pqr_cache[pdbid] = (charges_full, charges_nochain, has_chain)

    print(f"pdb2pqr complete: {len(pqr_cache)} ok, {len(pqr_failures)} failed")

    if pqr_failures:
        print("Failed PDB IDs:", pqr_failures[:20])

    return pqr_cache, pqr_failures

In [3]:
def clean_pdb_df_for_saving(df, record_name, start_atom_number=1, start_line_idx=0):
    """
    Cleans a Biopandas ATOM/HETATM dataframe so PandasPdb.to_pdb()
    does not fail from NaN values or wrong dtypes.
    """
    df = df.copy()
    df = df.dropna(how="all").copy()

    if df.empty:
        return df

    df = df.reset_index(drop=True)

    # Required record type
    df["record_name"] = record_name

    # Coordinates must be numeric
    coord_cols = ["x_coord", "y_coord", "z_coord"]
    for col in coord_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=coord_cols).copy()
    df = df.reset_index(drop=True)

    # Integer-like columns
    df["atom_number"] = np.arange(
        start_atom_number,
        start_atom_number + len(df)
    ).astype(int)

    if "residue_number" not in df.columns:
        df["residue_number"] = 1

    df["residue_number"] = pd.to_numeric(
        df["residue_number"],
        errors="coerce"
    ).fillna(1).astype(int)

    df["line_idx"] = np.arange(
        start_line_idx,
        start_line_idx + len(df)
    ).astype(int)

    # Float columns
    for col, default in {
        "x_coord": 0.0,
        "y_coord": 0.0,
        "z_coord": 0.0,
        "occupancy": 1.00,
        "b_factor": 0.00,
    }.items():
        if col not in df.columns:
            df[col] = default
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default).astype(float)

    # String columns
    string_defaults = {
        "atom_name": "X",
        "alt_loc": "",
        "residue_name": "UNK",
        "chain_id": "",
        "insertion": "",
        "segment_id": "",
        "element_symbol": "",
        "blank_1": "",
        "blank_2": "",
        "blank_3": "",
        "blank_4": "",
    }

    for col, default in string_defaults.items():
        if col not in df.columns:
            df[col] = default
        df[col] = df[col].fillna(default).astype(str)

    # Important:
    # Do NOT make charge an empty string.
    # Biopandas may expect this column to be float/NaN.
    if "charge" not in df.columns:
        df["charge"] = np.nan
    else:
        df["charge"] = pd.to_numeric(df["charge"], errors="coerce")

    # Clean atom names / element symbols
    df["atom_name"] = df["atom_name"].astype(str).str.strip()
    df["residue_name"] = df["residue_name"].astype(str).str.strip()
    df["chain_id"] = df["chain_id"].astype(str).str.strip()
    df["element_symbol"] = df["element_symbol"].astype(str).str.strip()

    # If element_symbol is missing, infer from atom_name
    missing_element = df["element_symbol"].eq("") | df["element_symbol"].eq("nan")
    df.loc[missing_element, "element_symbol"] = (
        df.loc[missing_element, "atom_name"]
        .str.extract(r"([A-Za-z]+)", expand=False)
        .fillna("C")
        .str[:2]
        .str.capitalize()
    )

    return df

def get_positive_ligand_atoms(positive_file, protein_name):
    protein_pdb_df = PandasPdb().read_pdb(positive_file)
    protein_pdb_df.df.keys()
    protein = protein_pdb_df.df['ATOM']
    protein = protein[~protein['atom_name'].str.startswith('H')] # don't use hydrogen
    protein_coords = protein[['x_coord', 'y_coord', 'z_coord']].values
    protein_centroid = protein_coords.mean(axis=0)
    print(set(protein['chain_id']))
    print(positive_file)

    ligand_df = PandasPdb().read_pdb(positive_file)
    ligand_df.df.keys()
    ligand = ligand_df.df['HETATM']
    ligand = ligand[ligand['residue_name']=="CLR"]
    x = list(set(zip(ligand['residue_number'], ligand['chain_id'])))

    #get the most inward residue
    min_distance = float('inf')
    closest_clr = None

    all_ligands = []

    for residue_number, chain_id in x:
        clr_atoms = ligand[(ligand['residue_number'] == residue_number) & (ligand['chain_id'] == chain_id)]
        if clr_atoms.empty:
            continue

        clr_coords = clr_atoms[['x_coord', 'y_coord', 'z_coord']].values
        clr_centroid = clr_coords.mean(axis=0)
        
        distance = np.linalg.norm(protein_centroid - clr_centroid)
        
        if distance < min_distance:
            min_distance = distance
            closest_clr = (residue_number, chain_id)

        grid_list_ = grid_list(clr_atoms)

        all_ligands.append(grid_list_)

    ligand_ = ligand[(ligand['residue_number'] == closest_clr[0]) & (ligand['chain_id'] == closest_clr[1])]
    grid_list_ = grid_list(ligand_)

    filtered_atoms = filtering_proteins(protein, grid_list_)

    # Save to pdb
    filtered_atoms_clean = clean_pdb_df_for_saving(
        filtered_atoms,
        record_name="ATOM",
        start_atom_number=1,
        start_line_idx=0
    )

    ligand_clean = clean_pdb_df_for_saving(
        ligand_,
        record_name="HETATM",
        start_atom_number=len(filtered_atoms_clean) + 1,
        start_line_idx=len(filtered_atoms_clean)
    )

    filtered_pdb = PandasPdb()
    filtered_pdb.df["ATOM"] = filtered_atoms_clean
    filtered_pdb.df["HETATM"] = ligand_clean

    filtered_pdb_path = f"filtered-opm-ivan-5A/positive/{protein_name}-filtered-with-ligand.pdb"
    os.makedirs(os.path.dirname(filtered_pdb_path), exist_ok=True)

    filtered_pdb.to_pdb(
        path=filtered_pdb_path,
        records=["ATOM", "HETATM"],
        gz=False,
        append_newline=True
    )

    print(f"Saved: {filtered_pdb_path}")

    return protein, all_ligands


In [4]:
def check_if_unlabeled_is_positive(positive_grid_list, unlabeled_grid_list, cutoff=5.0):
    positive_grid_list = np.asarray(positive_grid_list, dtype=float)
    unlabeled_grid_list = np.asarray(unlabeled_grid_list, dtype=float)

    if positive_grid_list.size == 0 or unlabeled_grid_list.size == 0:
        print("Empty positive_grid_list or unlabeled_grid_list")
        return False

    chol_centroid = positive_grid_list.mean(axis=0)
    vina_centroid = unlabeled_grid_list.mean(axis=0)

    distance = np.linalg.norm(chol_centroid - vina_centroid)
    print(f"Centroid distance: {distance:.3f} Å")
    return distance <= cutoff

def kabsch_align_coords(source_coords, target_coords):
    """
    Align source_coords onto target_coords using matched atoms.
    Returns rotation matrix R and translation vector t.
    """
    source_coords = np.asarray(source_coords, dtype=float)
    target_coords = np.asarray(target_coords, dtype=float)

    source_centroid = source_coords.mean(axis=0)
    target_centroid = target_coords.mean(axis=0)

    source_centered = source_coords - source_centroid
    target_centered = target_coords - target_centroid

    H = source_centered.T @ target_centered
    U, S, Vt = np.linalg.svd(H)

    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = target_centroid - source_centroid @ R

    return R, t


def transform_ligand_to_opm_coords(ligand_df, rcsb_protein_df, opm_protein_df):
    """
    Transform Vina/RCSB ligand coordinates into OPM coordinate space.
    Uses matched protein atoms to estimate the rigid-body transform.
    """
    rcsb = rcsb_protein_df.copy()
    opm = opm_protein_df.copy()

    rcsb["_key"] = make_atom_key(rcsb)
    opm["_key"] = make_atom_key(opm)

    rcsb = rcsb.drop_duplicates("_key")
    opm = opm.drop_duplicates("_key")

    merged = rcsb.merge(
        opm,
        on="_key",
        suffixes=("_rcsb", "_opm")
    )

    if len(merged) < 3:
        raise ValueError("Need at least 3 matched atoms to align RCSB ligand to OPM coordinates.")

    source_coords = merged[["x_coord_rcsb", "y_coord_rcsb", "z_coord_rcsb"]].values
    target_coords = merged[["x_coord_opm", "y_coord_opm", "z_coord_opm"]].values

    R, t = kabsch_align_coords(source_coords, target_coords)

    ligand = ligand_df.copy()
    ligand_coords = ligand[["x_coord", "y_coord", "z_coord"]].values.astype(float)
    ligand_coords_opm = ligand_coords @ R + t

    ligand["x_coord"] = ligand_coords_opm[:, 0]
    ligand["y_coord"] = ligand_coords_opm[:, 1]
    ligand["z_coord"] = ligand_coords_opm[:, 2]

    return ligand

In [5]:
# positive_files = glob.glob("OPM_PDB/*.pdb")
# positive_files = sorted(positive_files, key=natural_sort_key)

# unlabeled_files = glob.glob("CLR-Unlabeled-Distinct-matched-to-opm_ids/*.pdb")
# unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

# RCSB_PROTEIN_DIR = "CLR-PDB-matched-to-opm_ids"
# OPM_DIR = "OPM_PDB"

# for unlabeled_file in unlabeled_files:
#     unlabeled_name = get_protein_name(unlabeled_file)
#     fragment_index = get_mode_index(unlabeled_file)

#     if unlabeled_name is None:
#         print(f"Skipping file with no PDB id match: {unlabeled_file}")
#         continue

#     rcsb_file = os.path.join(RCSB_PROTEIN_DIR, f"{unlabeled_name}_protein.pdb")
#     opm_file = os.path.join(OPM_DIR, f"{unlabeled_name.lower()}.pdb")

#     if not os.path.exists(rcsb_file):
#         print(f"Missing RCSB file: {rcsb_file}")
#         continue

#     if not os.path.exists(opm_file):
#         print(f"Missing OPM file: {opm_file}")
#         continue

#     # OPM positive logic, keeps closest CLR and all_lig_gridlist
#     opm_protein, all_lig_gridlist = get_positive_ligand_atoms(opm_file, unlabeled_name)

#     # RCSB whole protein for filtering against Vina docked CLR
#     rcsb_pdb = PandasPdb().read_pdb(rcsb_file)
#     rcsb_protein = rcsb_pdb.df['ATOM']
#     rcsb_protein = rcsb_protein[~rcsb_protein['atom_name'].str.startswith('H')]

#     fragment_df = PandasPdb().read_pdb(unlabeled_file)
#     fragment = fragment_df.df['HETATM']
#     #fragment = fragment[fragment['residue_name'] == 'CLR']

#     grid_list_ = grid_list(fragment)

#     # Filter RCSB atoms using Vina CLR coordinates
#     filtered_rcsb_atoms = filtering_proteins(rcsb_protein, grid_list_)

#     if filtered_rcsb_atoms.empty:
#         print(f"No filtered RCSB atoms found for {unlabeled_file}")
#         continue

#     # Convert filtered RCSB atoms to OPM coordinates
#     filtered_opm_atoms = map_rcsb_atoms_to_opm(filtered_rcsb_atoms, opm_protein)

#     if filtered_opm_atoms.empty:
#         print(f"No OPM atom matches found for {unlabeled_file}")
#         continue

#     fragment_opm = transform_ligand_to_opm_coords(
#         ligand_df=fragment,
#         rcsb_protein_df=rcsb_protein,
#         opm_protein_df=opm_protein
#     )

#     filtered_opm_atoms_clean = clean_pdb_df_for_saving(
#         filtered_opm_atoms,
#         record_name="ATOM",
#         start_atom_number=1,
#         start_line_idx=0
#     )

#     fragment_opm_clean = clean_pdb_df_for_saving(
#         fragment_opm,
#         record_name="HETATM",
#         start_atom_number=len(filtered_opm_atoms_clean) + 1,
#         start_line_idx=len(filtered_opm_atoms_clean)
#     )

#     filtered_pdb = PandasPdb()
#     filtered_pdb.df["ATOM"] = filtered_opm_atoms_clean
#     filtered_pdb.df["HETATM"] = fragment_opm_clean

#     grid_list_filtered = grid_list(filtered_pdb.df['ATOM'])

#     is_positive = False
#     for lig in all_lig_gridlist:
#         if check_if_unlabeled_is_positive(lig.copy(), grid_list_filtered.copy()):
#             is_positive = True
#             break

#     if is_positive:
#         filtered_pdb_path = f"filtered-opm-vina-5A/unlabeled/{unlabeled_name}-f{fragment_index}-positive-with-ligand.pdb"
#     else:
#         filtered_pdb_path = f"filtered-opm-vina-5A/unlabeled/{unlabeled_name}-f{fragment_index}-with-ligand.pdb"

#     os.makedirs(os.path.dirname(filtered_pdb_path), exist_ok=True)
#     filtered_pdb.to_pdb(
#         path=filtered_pdb_path,
#         records=["ATOM", "HETATM"],
#         gz=False,
#         append_newline=True
#     )

#     print(f"Saved: {filtered_pdb_path}")

In [6]:
# import glob
# from pathlib import Path

# def pdb_id_from_file(path):
#     """
#     Extract PDB ID from file name.
#     Works for names like:
#     1abc.pdb
#     1ABC.pdb
#     1ABC_protein.pdb
#     1abc_opm.pdb
#     """
#     stem = Path(path).stem
#     stem = stem.replace("_protein", "")
#     return stem[:4].lower()


# # Ivan files are used only to get the PDB IDs
# ivan_files = glob.glob("/home/alexhernandez/p2rank/ivanfiles/*.pdb")

# ivan_pdb_ids = {
#     pdb_id_from_file(f)
#     for f in ivan_files
# }

# print("Number of Ivan PDB IDs:", len(ivan_pdb_ids))


# # Now use OPM PDB files, but only if their PDB ID is in ivanfiles
# opm_files = glob.glob("OPM_PDB/*.pdb")

# positive_files = [
#     f for f in opm_files
#     if pdb_id_from_file(f) in ivan_pdb_ids
# ]

# positive_files = sorted(positive_files, key=natural_sort_key)

# print("Number of matching OPM PDB files:", len(positive_files))


# # Optional: report Ivan IDs that do not have a matching OPM file
# opm_pdb_ids = {
#     pdb_id_from_file(f)
#     for f in opm_files
# }

# missing_from_opm = sorted(ivan_pdb_ids - opm_pdb_ids)

# print("Ivan PDB IDs missing from OPM_PDB:")
# print(missing_from_opm)


# # Run your normal positive logic, but now using matching OPM files
# for positive in positive_files:
#     positive_name = get_protein_name(positive)

#     protein, all_lig_gridlist = get_positive_ligand_atoms(
#         positive,
#         positive_name
#     )

In [7]:
import pickle
import os

PDB2PQR_PH = 7.4

cache_path = "pqr_cache.pkl"
FULL_PDB_DIR = "/home/alexhernandez/CholBindNet/OPM_GNN/Ligand_and_Protein/CLR-PDB-matched-to-opm_ids"   # change this

if os.path.exists(cache_path):
    with open(cache_path, "rb") as f:
        data = pickle.load(f)
        pqr_cache = data["pqr_cache"]
        pqr_failures = data["pqr_failures"]

    print(f"Loaded PQR cache from {cache_path}")
    print(f"Cached PDBs: {len(pqr_cache)}")
else:
    print("Cache not found, rebuilding...")
    pqr_cache, pqr_failures = build_pqr_cache(FULL_PDB_DIR, ph=PDB2PQR_PH)

Loaded PQR cache from pqr_cache.pkl
Cached PDBs: 763


In [8]:
print("num cached pdbs:", len(pqr_cache))
print("num failures:", len(pqr_failures))

example_key = next(iter(pqr_cache))
print("example pdb id:", example_key)

charges_full, charges_nochain, has_chain = pqr_cache[example_key]
print("has_chain:", has_chain)
print("num full charge entries:", len(charges_full))
print("num no-chain charge entries:", len(charges_nochain))

print("sample full keys:", list(charges_full.items())[:10])
print("sample no-chain keys:", list(charges_nochain.items())[:10])

num cached pdbs: 763
num failures: 7
example pdb id: 1LRI
has_chain: False
num full charge entries: 1432
num no-chain charge entries: 1432
sample full keys: [(('', 1, 'N'), 0.1812), (('', 1, 'CA'), 0.0034), (('', 1, 'C'), 0.6163), (('', 1, 'O'), -0.5722), (('', 1, 'CB'), 0.4514), (('', 1, 'OG1'), -0.6764), (('', 1, 'CG2'), -0.2554), (('', 1, 'H'), 0.1934), (('', 1, 'HA'), 0.1087), (('', 1, 'HB'), -0.0323)]
sample no-chain keys: [((1, 'N'), 0.1812), ((1, 'CA'), 0.0034), ((1, 'C'), 0.6163), ((1, 'O'), -0.5722), ((1, 'CB'), 0.4514), ((1, 'OG1'), -0.6764), ((1, 'CG2'), -0.2554), ((1, 'H'), 0.1934), ((1, 'HA'), 0.1087), ((1, 'HB'), -0.0323)]


In [9]:
# -------------------------
# Atom subtype setup from second notebook + OPM z-depth support
# -------------------------
ATOM_SUBTYPES = [
    # Carbon (17)
    "C", "CA", "CB", "CD", "CD1", "CD2", "CE", "CE1", "CE2", "CE3",
    "CG", "CG1", "CG2", "CH2", "CZ", "CZ2", "CZ3",

    # Oxygen (8)
    "O", "OH", "OD1", "OD2", "OE1", "OE2", "OG", "OG1",

    # Nitrogen (9)
    "N", "NE", "NE1", "NE2", "ND1", "ND2", "NZ", "NH1", "NH2",

    # Sulfur (2)
    "SD", "SG",

    # Unknown / non-standard atom names
    "UNKNOWN",
]

ATOM_TO_INDEX = {atom_name: idx for idx, atom_name in enumerate(ATOM_SUBTYPES)}

STANDARD_RESIDUES = [
    "ALA", "ARG", "ASN", "ASP", "CYS",
    "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO",
    "SER", "THR", "TRP", "TYR", "VAL",
]

RESIDUE_FEATURE_NAMES = [
    f"residue_{resname}" for resname in STANDARD_RESIDUES
]

RDKIT_FEATURE_NAMES = [
    # 37 atom-name subtype one-hot features
    *ATOM_SUBTYPES,

    # RDKit / structure features
    "degree_norm",

    "hybridization_SP",
    "hybridization_SP2",
    "hybridization_SP3",

    # Normalized or raw partial charge value
    "partial_charge",

    # One-hot charge sign features from RAW partial charge
    "partial_charge_positive",
    "partial_charge_negative",

    "is_in_ring",
    "is_aromatic",

    # 20 separate amino acid residue one-hot columns
    *RESIDUE_FEATURE_NAMES,
]

FEATURE_INDEX = {name: idx for idx, name in enumerate(RDKIT_FEATURE_NAMES)}
LIGAND_RESNAMES = {"CLR", "CHL", "UNL"}

def compute_weighted_inverse_distance(
    coords,
    protein_mask,
    ligand_mask,
    pp_weight=2.0,
    pl_weight=1.0,
    ll_weight=0.5,
    row_normalize=True
):
    """
    Builds inverse-distance matrix with edge-type weights.

    pp_weight: protein-protein edge importance
    pl_weight: protein-ligand edge importance
    ll_weight: ligand-ligand edge importance
    """
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff ** 2, axis=-1))

    with np.errstate(divide="ignore"):
        inv = 1.0 / distances

    np.fill_diagonal(inv, 1.0)
    inv = np.minimum(inv, 1.0)

    weights = np.ones_like(inv, dtype=np.float32) * pl_weight

    pp_edges = np.outer(protein_mask, protein_mask)
    ll_edges = np.outer(ligand_mask, ligand_mask)

    weights[pp_edges] = pp_weight
    weights[ll_edges] = ll_weight

    weighted_inv = inv * weights

    np.fill_diagonal(weighted_inv, 1.0)

    if row_normalize:
        row_sums = weighted_inv.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        weighted_inv = weighted_inv / row_sums

    return weighted_inv.astype(np.float32)

def atom_name_to_subtype(atomname, atom_symbol=None):
    atomname = str(atomname).strip().upper()

    if atomname in ATOM_TO_INDEX:
        return atomname

    return "UNKNOWN"

def compute_z_depth_from_coords(coords):
    """
    Normalized signed z-depth using fixed membrane range [-50, 50].

    Mapping:
    z <= -50 -> 0
    z = 0    -> 0.5
    z >= 50  -> 1
    """
    z = coords[:, 2].astype(float)

    # Cap z values to [-50, 50]
    z_clipped = np.clip(z, -50, 50)

    # Normalize from [-50, 50] to [0, 1]
    z_norm = (z_clipped + 50) / 100

    return z_norm.reshape(-1, 1).astype(np.float32)

def rdkit_features_with_atom_subtypes(
    pdb_path,
    pqr_cache=None,
    charge_min=-1.003100,
    charge_max=0.885100,
    normalize_charge=True,
    ligand_feature_mode="full",
    opm_dir=OPM_DIR,
    return_atom_info=False,
):
    """
    Builds RDKit-style features with atom subtype one-hot columns.

    Returns:
        coords:       (N, 3)
        features:     (N, 67) by default:
                        37 atom subtypes
                        + degree_norm
                        + 3 hybridization columns
                        + partial_charge
                        + 2 raw-charge sign one-hot columns
                        + ring + aromatic
                        + 20 individual residue one-hot columns
                        + z_depth
        protein_mask: (N,)
        ligand_mask:  (N,)
        z_depth:      (N, 1)

    ligand_feature_mode:
        "full"   -> keep all ligand features
        "none"   -> ligand feature rows are all zero, but ligand atoms remain in distance matrix
        "z_only" -> ligand rows only keep z-depth
    """
    mol = Chem.MolFromPDBFile(
        str(pdb_path),
        removeHs=False,
        sanitize=False,
    )

    if mol is None:
        print(f"[SKIP] RDKit could not parse: {pdb_path}")
        empty_return = (None, None, None, None, None)
        if return_atom_info:
            return (*empty_return, None)
        return empty_return

    try:
        Chem.SanitizeMol(mol)
    except Exception:
        print(f"[WARN] RDKit sanitize warning for: {pdb_path}")

    conf = mol.GetConformer()

    heavy_atoms = [
        atom for atom in mol.GetAtoms()
        if atom.GetAtomicNum() != 1
    ]

    n_atoms = len(heavy_atoms)
    n_features = len(RDKIT_FEATURE_NAMES)

    coords = np.zeros((n_atoms, 3), dtype=np.float32)
    features = np.zeros((n_atoms, n_features), dtype=np.float32)

    protein_mask = np.zeros(n_atoms, dtype=bool)
    ligand_mask = np.zeros(n_atoms, dtype=bool)

    atom_info_rows = []

    base_name = os.path.basename(str(pdb_path))
    match = re.match(r"([A-Za-z0-9]{4})", base_name)
    pdb_id = match.group(1).upper() if match else None

    charges_full, charges_nochain, has_chain = ({}, {}, False)
    if pqr_cache is not None and pdb_id in pqr_cache:
        charges_full, charges_nochain, has_chain = pqr_cache[pdb_id]

    idx = FEATURE_INDEX

    for i, atom in enumerate(heavy_atoms):
        pos = conf.GetAtomPosition(atom.GetIdx())
        coords[i] = [pos.x, pos.y, pos.z]

        pdb_info = atom.GetPDBResidueInfo()

        if pdb_info is not None:
            residue = pdb_info.GetResidueName().strip()
            atomname = pdb_info.GetName().strip()
            chain = (pdb_info.GetChainId() or "").strip()
            resseq = int(pdb_info.GetResidueNumber())
        else:
            residue = ""
            atomname = atom.GetSymbol()
            chain = ""
            resseq = -9999

        atom_symbol = atom.GetSymbol().upper()
        atom_subtype = atom_name_to_subtype(atomname, atom_symbol=atom_symbol)

        is_ligand = residue in LIGAND_RESNAMES
        ligand_mask[i] = is_ligand
        protein_mask[i] = not is_ligand

        # 37 atom subtype one-hot columns
        features[i, idx[atom_subtype]] = 1.0

        # Degree feature kept from the OPM notebook
        features[i, idx["degree_norm"]] = min(atom.GetDegree() / 4.0, 1.0)

        # Hybridization features
        hybridization = atom.GetHybridization()
        if hybridization == Chem.HybridizationType.SP:
            features[i, idx["hybridization_SP"]] = 1.0
        elif hybridization == Chem.HybridizationType.SP2:
            features[i, idx["hybridization_SP2"]] = 1.0
        elif hybridization == Chem.HybridizationType.SP3:
            features[i, idx["hybridization_SP3"]] = 1.0

        # Partial charge from PQR cache
        partial_charge_raw = lookup_charge(
            chain,
            resseq,
            atomname,
            charges_full,
            charges_nochain,
            has_chain,
        )

        # Charge sign one-hot columns should use RAW charge,
        # not normalized charge.
        if partial_charge_raw > 0:
            features[i, idx["partial_charge_positive"]] = 1.0
        elif partial_charge_raw < 0:
            features[i, idx["partial_charge_negative"]] = 1.0

        partial_charge = partial_charge_raw

        if normalize_charge:
            if charge_max == charge_min:
                partial_charge = 0.0
            else:
                partial_charge = (partial_charge - charge_min) / (charge_max - charge_min)
                partial_charge = np.clip(partial_charge, 0.0, 1.0)

        features[i, idx["partial_charge"]] = float(partial_charge)

        # Ring/aromatic flags
        features[i, idx["is_in_ring"]] = 1.0 if atom.IsInRing() else 0.0
        features[i, idx["is_aromatic"]] = 1.0 if atom.GetIsAromatic() else 0.0

        # Individual residue one-hot features
        # Protein residues get one column per amino acid.
        # Ligands such as CLR/CHL/UNL will not activate these residue columns.
        if residue in STANDARD_RESIDUES:
            features[i, idx[f"residue_{residue}"]] = 1.0

        atom_info_rows.append({
            "row_index": i,
            "atom_name": atomname,
            "atom_subtype": atom_subtype,
            "atom_symbol": atom_symbol,
            "residue_name": residue,
            "residue_number": resseq,
            "chain_id": chain,
            "is_protein": bool(protein_mask[i]),
            "is_ligand": bool(ligand_mask[i]),
            "x": coords[i, 0],
            "y": coords[i, 1],
            "z": coords[i, 2],
            "partial_charge_raw": float(partial_charge_raw),
            "partial_charge_normalized": float(partial_charge),
        })

    z_depth = compute_z_depth_from_coords(coords)

    if ligand_feature_mode == "none":
        features[ligand_mask, :] = 0.0

    elif ligand_feature_mode == "full":
        pass

    else:
        raise ValueError("ligand_feature_mode must be 'full', 'none', or 'z_only'")

    if return_atom_info:
        atom_info_df = pd.DataFrame(atom_info_rows)
        return coords, features, protein_mask, ligand_mask, z_depth, atom_info_df

    return coords, features, protein_mask, ligand_mask, z_depth


# Keep the old function name so the rest of your notebook cells do not need changes.
rdkit_features_no_atom_subtypes = rdkit_features_with_atom_subtypes

In [10]:
# max_atoms = 200
# output_dir = "cholesterol-rdkit-opm-ligandgraph-5A/positive"
# os.makedirs(output_dir, exist_ok=True)

# positive_files = glob.glob("filtered-opm-vina-5A/positive/*with-ligand.pdb")
# positive_files = sorted(positive_files, key=natural_sort_key)

# LIGAND_Z_MIN = -35.0
# LIGAND_Z_MAX = 35.0

# skipped_by_ligand_z = []

# for file in positive_files:
#     coords, encoded_matrix, protein_mask, ligand_mask, z_depth = rdkit_features_no_atom_subtypes(
#         file,
#         pqr_cache=pqr_cache,
#         charge_min=-1.003100,
#         charge_max=0.885100,
#         normalize_charge=True,
#         ligand_feature_mode="full"
#     )

#     if coords is None or encoded_matrix is None:
#         print(f"[SKIP] Failed encoding for {file}")
#         continue

#     # --------------------------------------------------
#     # Skip positive samples where any ligand atom is
#     # outside the allowed z range [-35, 35]
#     # --------------------------------------------------
#     ligand_z = coords[ligand_mask, 2]

#     if ligand_z.size == 0:
#         print(f"[SKIP] No ligand atoms found in {file}")
#         skipped_by_ligand_z.append({
#             "file": file,
#             "reason": "no_ligand_atoms",
#             "ligand_z_min": np.nan,
#             "ligand_z_max": np.nan,
#             "num_ligand_atoms": 0,
#         })
#         continue

#     ligand_z_min = ligand_z.min()
#     ligand_z_max = ligand_z.max()

#     if ligand_z_min < LIGAND_Z_MIN or ligand_z_max > LIGAND_Z_MAX:
#         print(
#             f"[SKIP] Ligand z outside [{LIGAND_Z_MIN}, {LIGAND_Z_MAX}] for {file}: "
#             f"min={ligand_z_min:.3f}, max={ligand_z_max:.3f}"
#         )

#         skipped_by_ligand_z.append({
#             "file": file,
#             "reason": "ligand_z_out_of_range",
#             "ligand_z_min": float(ligand_z_min),
#             "ligand_z_max": float(ligand_z_max),
#             "num_ligand_atoms": int(ligand_z.size),
#         })
#         continue

#     weighted_inverse_distance = compute_weighted_inverse_distance(
#         coords,
#         protein_mask=protein_mask,
#         ligand_mask=ligand_mask,
#         pp_weight=2.0,
#         pl_weight=1.0,
#         ll_weight=0.5,
#         row_normalize=True
#     )

#     if weighted_inverse_distance.shape[0] != encoded_matrix.shape[0]:
#         raise ValueError(
#             f"Atom mismatch in {file}: "
#             f"weighted_inverse_distance={weighted_inverse_distance.shape}, "
#             f"encoded_matrix={encoded_matrix.shape}"
#         )

#     combined_matrix = weighted_inverse_distance @ encoded_matrix
#     combined_matrix = combined_matrix * z_depth

#     num_atoms = combined_matrix.shape[0]

#     if num_atoms > max_atoms:
#         print(f"[SKIP] {file} has {num_atoms} atoms, exceeding max_atoms={max_atoms}")
#         continue

#     combined_matrix_padded = np.pad(
#         combined_matrix,
#         ((0, max_atoms - num_atoms), (0, 0)),
#         mode="constant"
#     )

#     protein_mask_padded = np.pad(
#         protein_mask.astype(np.int8),
#         (0, max_atoms - num_atoms),
#         mode="constant"
#     )

#     ligand_mask_padded = np.pad(
#         ligand_mask.astype(np.int8),
#         (0, max_atoms - num_atoms),
#         mode="constant"
#     )

#     z_depth_padded = np.pad(
#         z_depth,
#         ((0, max_atoms - num_atoms), (0, 0)),
#         mode="constant"
#     )

#     base_name = os.path.splitext(os.path.basename(file))[0]

#     matrix_path = os.path.join(output_dir, f"{base_name}_combined_matrix.npy")
#     meta_path = os.path.join(output_dir, f"{base_name}_metadata.npz")

#     np.save(matrix_path, combined_matrix_padded)

#     np.savez(
#         meta_path,
#         protein_mask=protein_mask_padded,
#         ligand_mask=ligand_mask_padded,
#         z_depth=z_depth_padded,
#         feature_names=np.array(RDKIT_FEATURE_NAMES),
#         pp_weight=np.array([2.0]),
#         pl_weight=np.array([1.0]),
#         ll_weight=np.array([0.5])
#     )


# skipped_by_ligand_z_df = pd.DataFrame(skipped_by_ligand_z)

# print(f"Skipped positive samples by ligand z filter: {len(skipped_by_ligand_z_df)}")

# if not skipped_by_ligand_z_df.empty:
#     display(skipped_by_ligand_z_df)

In [11]:
max_atoms = 200
output_dir = "cholesterol-rdkit-ivan-ligandgraph-5A/positive"
os.makedirs(output_dir, exist_ok=True)

positive_files = glob.glob("filtered-opm-ivan-5A/positive/*with-ligand.pdb")
positive_files = sorted(positive_files, key=natural_sort_key)

for file in positive_files:
    coords, encoded_matrix, protein_mask, ligand_mask, z_depth = rdkit_features_no_atom_subtypes(
        file,
        pqr_cache=pqr_cache,
        charge_min=-1.003100,
        charge_max=0.885100,
        normalize_charge=True,
        ligand_feature_mode="full"
    )

    if coords is None or encoded_matrix is None:
        print(f"[SKIP] Failed encoding for {file}")
        continue

    weighted_inverse_distance = compute_weighted_inverse_distance(
        coords,
        protein_mask=protein_mask,
        ligand_mask=ligand_mask,
        pp_weight=2.0,
        pl_weight=1.0,
        ll_weight=0.5,
        row_normalize=True
    )

    if weighted_inverse_distance.shape[0] != encoded_matrix.shape[0]:
        raise ValueError(
            f"Atom mismatch in {file}: "
            f"weighted_inverse_distance={weighted_inverse_distance.shape}, "
            f"encoded_matrix={encoded_matrix.shape}"
        )

    combined_matrix = weighted_inverse_distance @ encoded_matrix
    combined_matrix = combined_matrix * z_depth

    num_atoms = combined_matrix.shape[0]

    if num_atoms > max_atoms:
        print(f"[SKIP] {file} has {num_atoms} atoms, exceeding max_atoms={max_atoms}")
        continue

    combined_matrix_padded = np.pad(
        combined_matrix,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode="constant"
    )

    protein_mask_padded = np.pad(
        protein_mask.astype(np.int8),
        (0, max_atoms - num_atoms),
        mode="constant"
    )

    ligand_mask_padded = np.pad(
        ligand_mask.astype(np.int8),
        (0, max_atoms - num_atoms),
        mode="constant"
    )

    z_depth_padded = np.pad(
        z_depth,
        ((0, max_atoms - num_atoms), (0, 0)),
        mode="constant"
    )

    base_name = os.path.splitext(os.path.basename(file))[0]

    matrix_path = os.path.join(output_dir, f"{base_name}_combined_matrix.npy")
    meta_path = os.path.join(output_dir, f"{base_name}_metadata.npz")

    np.save(matrix_path, combined_matrix_padded)

    np.savez(
        meta_path,
        protein_mask=protein_mask_padded,
        ligand_mask=ligand_mask_padded,
        z_depth=z_depth_padded,
        feature_names=np.array(RDKIT_FEATURE_NAMES),
        pp_weight=np.array([2.0]),
        pl_weight=np.array([1.0]),
        ll_weight=np.array([0.5])
    )

[SKIP] RDKit could not parse: filtered-opm-ivan-5A/positive/6AWN-filtered-with-ligand.pdb
[SKIP] Failed encoding for filtered-opm-ivan-5A/positive/6AWN-filtered-with-ligand.pdb
[SKIP] RDKit could not parse: filtered-opm-ivan-5A/positive/6AWO-filtered-with-ligand.pdb
[SKIP] Failed encoding for filtered-opm-ivan-5A/positive/6AWO-filtered-with-ligand.pdb
[SKIP] RDKit could not parse: filtered-opm-ivan-5A/positive/6AWQ-filtered-with-ligand.pdb
[SKIP] Failed encoding for filtered-opm-ivan-5A/positive/6AWQ-filtered-with-ligand.pdb
[SKIP] RDKit could not parse: filtered-opm-ivan-5A/positive/6JXK-filtered-with-ligand.pdb
[SKIP] Failed encoding for filtered-opm-ivan-5A/positive/6JXK-filtered-with-ligand.pdb


[10:06:07] 

****
Post-condition Violation
Element 'Cb' not found
Violation occurred on line 93 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/PeriodicTable.h
Failed Expression: anum > -1
----------
Stacktrace:
----------
****

[10:06:07] 

****
Post-condition Violation
Element 'Cb' not found
Violation occurred on line 93 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/PeriodicTable.h
Failed Expression: anum > -1
----------
Stacktrace:
----------
****

[10:06:07] 

****
Post-condition Violation
Element 'Cb' not found
Violation occurred on line 93 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/PeriodicTable.h
Failed Expression: anum > -1
----------
Stacktrace:
----------
****

[10:06:07] 

****
Post-condition Violation
Element 'Cb' not found
Violation occurred on line 93 in file /project/build/temp.linux-x86_64-cpython-312/rdkit/Code/GraphMol/PeriodicTable.h
Failed Expression: anum > -1
----------
Stacktrace:
-

In [12]:
def inspect_preprocessed_sample(
    pdb_path,
    combined_matrix_path,
    metadata_path,
    pqr_cache=None
):
    """
    Inspect one filtered PDB and its preprocessed matrix/metadata.

    Important:
    - encoded_matrix = raw atom features before graph aggregation
    - combined_matrix = weighted_inverse_distance @ encoded_matrix
    """

    pdb_path = str(pdb_path)
    combined_matrix_path = str(combined_matrix_path)
    metadata_path = str(metadata_path)

    combined_matrix = np.load(combined_matrix_path)
    metadata = np.load(metadata_path, allow_pickle=True)

    saved_protein_mask = metadata["protein_mask"].astype(bool)
    saved_ligand_mask = metadata["ligand_mask"].astype(bool)
    saved_feature_names = metadata["feature_names"]

    coords, encoded_matrix, protein_mask, ligand_mask, z_depth, atom_info_df = rdkit_features_no_atom_subtypes(
        pdb_path,
        pqr_cache=pqr_cache,
        charge_min=-1.003100,
        charge_max=0.885100,
        normalize_charge=True,
        ligand_feature_mode="full",
        return_atom_info=True
    )

    if coords is None:
        print("Could not encode PDB.")
        return None, None, None

    num_real_atoms = len(encoded_matrix)

    print("========== SHAPES ==========")
    print("Raw encoded_matrix shape:", encoded_matrix.shape)
    print("Saved combined_matrix shape:", combined_matrix.shape)
    print("Number of real atoms from PDB:", num_real_atoms)
    print("Number of saved protein atoms:", saved_protein_mask.sum())
    print("Number of saved ligand atoms:", saved_ligand_mask.sum())
    print("Number of saved padding rows:", len(saved_protein_mask) - saved_protein_mask.sum() - saved_ligand_mask.sum())

    print("\n========== FEATURE NAMES ==========")
    for i, name in enumerate(saved_feature_names):
        print(f"{i}: {name}")

    raw_feature_df = pd.DataFrame(
        encoded_matrix,
        columns=RDKIT_FEATURE_NAMES
    )

    combined_real_df = pd.DataFrame(
        combined_matrix[:num_real_atoms],
        columns=saved_feature_names
    )

    atom_raw_df = pd.concat(
        [atom_info_df.reset_index(drop=True), raw_feature_df.reset_index(drop=True)],
        axis=1
    )

    atom_combined_df = pd.concat(
        [atom_info_df.reset_index(drop=True), combined_real_df.reset_index(drop=True)],
        axis=1
    )

    protein_raw = atom_raw_df[atom_raw_df["is_protein"]].copy()
    ligand_raw = atom_raw_df[atom_raw_df["is_ligand"]].copy()

    protein_combined = atom_combined_df[atom_combined_df["is_protein"]].copy()
    ligand_combined = atom_combined_df[atom_combined_df["is_ligand"]].copy()

    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.width", None,
        "display.max_colwidth", None,
        "display.float_format", "{:.6f}".format
    ):
        print("\n========== RAW PROTEIN FEATURES ==========")
        display(protein_raw)

        print("\n========== RAW LIGAND FEATURES ==========")
        display(ligand_raw)

        print("\n========== COMBINED PROTEIN FEATURES ==========")
        display(protein_combined)

        print("\n========== COMBINED LIGAND FEATURES ==========")
        display(ligand_combined)

    return atom_raw_df, atom_combined_df, metadata

pdb_path = "/home/alexhernandez/CholBindNet/OPM_GNN/Ligand_and_Protein/filtered-opm-vina-5A/unlabeled/1ZHY-f1-with-ligand.pdb"

combined_matrix_path = "/home/alexhernandez/CholBindNet/OPM_GNN/Ligand_and_Protein/cholesterol-rdkit-opm-ligandgraph-5A/unlabeled/1ZHY-f1-with-ligand_combined_matrix.npy"

metadata_path = "/home/alexhernandez/CholBindNet/OPM_GNN/Ligand_and_Protein/cholesterol-rdkit-opm-ligandgraph-5A/unlabeled/1ZHY-f1-with-ligand_metadata.npz"

atom_raw_df, atom_combined_df, metadata = inspect_preprocessed_sample(
    pdb_path=pdb_path,
    combined_matrix_path=combined_matrix_path,
    metadata_path=metadata_path,
    pqr_cache=pqr_cache,
)

FileNotFoundError: [Errno 2] No such file or directory: '/home/alexhernandez/CholBindNet/OPM_GNN/Ligand_and_Protein/cholesterol-rdkit-opm-ligandgraph-5A/unlabeled/1ZHY-f1-with-ligand_combined_matrix.npy'